## 1. Imports
Loading all the required libraries for our time series analysis and modeling.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima

# Make plots visible inside notebook
%matplotlib inline


## 2. Load Dataset (CSV)
Loading the raw food inflation data from the World Bank dataset.


In [ ]:
file_path = "C:\\Users\\LENOVO\\OneDrive\\Desktop\\ESI\\Data csvs\\food_inflation.csv"
df_raw = pd.read_csv(file_path, skiprows=4)
df_raw.columns = df_raw.columns.str.strip()
print(df_raw.head())


## 3. Data Cleaning & Formatting
Filtering the data for India, removing unnecessary columns, reshaping the dataset (melt), and setting the Year as a datetime index object.


In [ ]:
df_india = df_raw[df_raw["Country Name"] == "India"]

df_india = df_india.drop(
    columns=["Country Name", "Country Code",
             "Indicator Name", "Indicator Code"],
    errors="ignore"
)

df_india = df_india.melt(
    var_name="Year",
    value_name="Food_Inflation"
)

df_india["Year"] = df_india["Year"].astype(int)
df_india = df_india.sort_values("Year").reset_index(drop=True)

df_india["Year"] = pd.to_datetime(df_india["Year"], format="%Y")
df_india = df_india.set_index("Year")

print(df_india.head())


## 4. Exploratory Data Analysis (plots)
Visualizing the historical food inflation over time.


In [ ]:
df_india.plot(figsize=(10,5))
plt.title("India Food Inflation (1991–2024)")
plt.show()


## 5. Stationarity Testing (ADF test)
Using the Augmented Dickey-Fuller test to determine if the series is stationary.


In [ ]:
result = adfuller(df_india["Food_Inflation"])

print("ADF Statistic:", result[0])
print("p-value:", result[1])


## 6. Differencing (if needed)
If the series is non-stationary, difference the data to remove trends and achieve stationarity.


In [ ]:
# Example of 1st order differencing
df_india_diff = df_india["Food_Inflation"].diff().dropna()

result_diff = adfuller(df_india_diff)
print("ADF Statistic (Differenced):", result_diff[0])
print("p-value (Differenced):", result_diff[1])


## 7. ACF and PACF Analysis
Plotting Autocorrelation and Partial Autocorrelation to help estimate initial AR (p) and MA (q) terms.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df_india["Food_Inflation"].dropna(), ax=axes[0])
plot_pacf(df_india["Food_Inflation"].dropna(), ax=axes[1])
plt.show()


## 8. Train / Test Split
Splitting chronologically to perform backtesting.
- Train: 1991–2018
- Test: 2019–2024


In [ ]:
train = df_india.loc[:'2018']
test = df_india.loc['2019':]

print(train.shape)
print(test.shape)


## 9. Auto ARIMA Model Selection
Using Auto-ARIMA to search the parameter space (p, d, q) automatically and pick the model with the lowest AIC score.


In [ ]:
auto_model = auto_arima(
    train["Food_Inflation"],
    start_p=0,
    start_q=0,
    max_p=5,
    max_q=5,
    d=1,
    seasonal=False,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

print(auto_model.summary())


## 10. Train Final ARIMA Model
Training the ARIMA model based on the best order returned by Auto-ARIMA.


In [ ]:
# Training using order=(2,1,1) as evaluated previously
model = ARIMA(train["Food_Inflation"], order=(2,1,1))
results = model.fit()

print(results.summary())


## 11. Forecast (2019–2024)
Predicting values for the duration matching our test subset length.


In [ ]:
forecast = results.forecast(steps=len(test))


## 12. Model Evaluation (MAE, RMSE)
Evaluating the forecast accuracy against actual observations using MAE (Mean Absolute Error) and RMSE (Root Mean Squared Error).


In [ ]:
comparison = test.copy()
comparison["Predicted"] = forecast
print(comparison)

mae = mean_absolute_error(comparison["Food_Inflation"], comparison["Predicted"])
rmse = np.sqrt(mean_squared_error(comparison["Food_Inflation"], comparison["Predicted"]))

print("MAE:", mae)
print("RMSE:", rmse)


## 13. Visualization (Actual vs Predicted)
Plotting the train data, actual test data, and out-of-sample predictions over time.


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(train.index, train["Food_Inflation"], label="Training Data")
plt.plot(test.index, test["Food_Inflation"], label="Actual Data")
plt.plot(test.index, forecast, label="Predicted", linestyle="--")
plt.legend()
plt.title("Food Inflation Forecast (ARIMA)")
plt.show()


## 14. Save Results
Exporting the cleaned dataset and the final forecast predictions into CSV files.


In [ ]:
df_india.to_csv("india_food_inflation_clean.csv")
comparison.to_csv("food_inflation_predictions.csv")


## 15. Residual Diagnostics
Checking if the residuals behave like random noise with no visible pattern.


In [ ]:
residuals = results.resid

import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.plot(residuals)
plt.title("Residuals of ARIMA Model")
plt.show()

## 16. ESI Report Component: Food Inflation

### Dataset
- **Indicator:** India Food Inflation
- **Period:** 1991–2024
- **Source:** World Bank (and/or RBI / MOSPI)

### Model
- **Algorithm:** ARIMA(p,d,q)
- **Selection Method:** Selected via Auto-ARIMA

### Evaluation
- **Metrics:** MAE, RMSE
- **Backtest Period:** 2019–2024

### Visualization
- **Actual vs Forecast:** To demonstrate model accuracy and trend capture.
- **Residual Plot:** To ensure errors resemble random white noise with no visible patterns.